<a href="https://colab.research.google.com/github/NABI-SNU/book/blob/main/tutorials/Session_1_DeepLearning/student/Tutorial2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Tutorial 2: Training Deep Networks

**Session 1: Deep Learning Foundations**

**Objective:** Understand how neural networks are optimized in practice.


## Tutorial Objectives

Deep networks are trained by minimizing empirical risk:

$$
\mathcal{L}(\theta) = \frac{1}{N}\sum_{i=1}^{N} \ell(f_\theta(\mathbf{x}_i), y_i).
$$

Gradient descent updates parameters in the direction that reduces the loss:

$$
\theta \leftarrow \theta - \eta \nabla_\theta \mathcal{L}(\theta).
$$

In this tutorial, we will implement training with mini-batch SGD, track training and validation loss, and observe the effect of learning rate.


In [ ]:
# Imports and shared settings
import random
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

SEED = 4
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

plt.rcParams['figure.figsize'] = (6, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False


In [ ]:
def get_mnist_loaders(batch_size=128, train_subset=12000, val_size=2000):
    """Return small MNIST train/validation/test loaders for quick tutorials."""
    transform = transforms.ToTensor()

    full_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

    if train_subset is not None:
        indices = torch.randperm(len(full_train))[:train_subset + val_size]
        full_train = Subset(full_train, indices)

    train_size = len(full_train) - val_size
    train_data, val_data = random_split(
        full_train,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(SEED),
    )

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = get_mnist_loaders()
images, labels = next(iter(train_loader))
print('Image batch:', images.shape)
print('Label batch:', labels.shape)


## Forward Pass, Loss, Backward Pass, Update

A single training step has four important pieces:

1. Run the model forward to compute logits.
2. Compute the loss.
3. Use backpropagation to compute gradients.
4. Update parameters with an optimizer.


In [ ]:
class SmallMLP(nn.Module):
    def __init__(self, hidden_units=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, 10),
        )

    def forward(self, x):
        return self.net(x)

model = SmallMLP().to(DEVICE)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

images, labels = next(iter(train_loader))
images = images.to(DEVICE)
labels = labels.to(DEVICE)

logits = model(images)          # forward pass
loss = loss_fn(logits, labels)  # empirical loss on one mini-batch
optimizer.zero_grad()           # clear stale gradients
loss.backward()                 # backpropagation
optimizer.step()                # parameter update

print('Mini-batch loss:', loss.item())
print('Logit shape:', logits.shape)


In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        logits = model(images)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += images.size(0)

    return total_loss / total_examples, total_correct / total_examples


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(images)
            loss = loss_fn(logits, labels)

            total_loss += loss.item() * images.size(0)
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_examples += images.size(0)

    return total_loss / total_examples, total_correct / total_examples


def fit(model, train_loader, val_loader, n_epochs=5, lr=1e-2, momentum=0.0):
    model = model.to(DEVICE)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(n_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(
            f'Epoch {epoch + 1:02d} | '
            f'train loss {train_loss:.3f}, acc {train_acc:.3f} | '
            f'val loss {val_loss:.3f}, acc {val_acc:.3f}'
        )

    return history


def plot_history(history, title='Training curves'):
    epochs = np.arange(1, len(history['train_loss']) + 1)
    fig, axs = plt.subplots(1, 2, figsize=(11, 4))

    axs[0].plot(epochs, history['train_loss'], marker='o', label='train')
    axs[0].plot(epochs, history['val_loss'], marker='o', label='validation')
    axs[0].set_xlabel('Epoch')
    axs[0].set_ylabel('Cross-entropy loss')
    axs[0].set_title('Loss')
    axs[0].legend()

    axs[1].plot(epochs, history['train_acc'], marker='o', label='train')
    axs[1].plot(epochs, history['val_acc'], marker='o', label='validation')
    axs[1].set_xlabel('Epoch')
    axs[1].set_ylabel('Accuracy')
    axs[1].set_ylim(0, 1)
    axs[1].set_title('Accuracy')
    axs[1].legend()

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


## Train and Validate

The training set updates the parameters. The validation set is held out from gradient descent and helps us detect overfitting or bad hyperparameters.


In [ ]:
model = SmallMLP(hidden_units=128)
history = fit(model, train_loader, val_loader, n_epochs=8, lr=0.1)
plot_history(history, title='Training and validation curves')


## Exercise: Learning Rate and Convergence

The learning rate controls the update size. Too small can make learning slow; too large can make training unstable.

Run the next cell and compare validation curves.


In [ ]:
learning_rates = [0.001, 0.01, 0.1, 1.0]
lr_histories = {}

for lr in learning_rates:
    print(f'\nTraining with learning rate {lr}')
    torch.manual_seed(SEED)
    model = SmallMLP(hidden_units=128)
    lr_histories[lr] = fit(model, train_loader, val_loader, n_epochs=5, lr=lr)


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for lr, hist in lr_histories.items():
    epochs = np.arange(1, len(hist['train_loss']) + 1)
    axs[0].plot(epochs, hist['train_loss'], marker='o', label=f'lr={lr}')
    axs[1].plot(epochs, hist['val_loss'], marker='o', label=f'lr={lr}')

axs[0].set_title('Training loss')
axs[1].set_title('Validation loss')
for ax in axs:
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Cross-entropy loss')
    ax.legend()
plt.tight_layout()
plt.show()


## Inspect Gradient Flow

Gradients tell each parameter how the loss would change if that parameter changed. Very small gradients can slow learning; very large gradients can make updates unstable.


In [ ]:
def gradient_norms(model):
    norms = {}
    for name, param in model.named_parameters():
        if param.grad is not None:
            norms[name] = param.grad.norm().item()
    return norms

model = SmallMLP().to(DEVICE)
loss_fn = nn.CrossEntropyLoss()
images, labels = next(iter(train_loader))
images = images.to(DEVICE)
labels = labels.to(DEVICE)

loss = loss_fn(model(images), labels)
model.zero_grad()
loss.backward()

for name, norm in gradient_norms(model).items():
    print(f'{name:20s} gradient norm: {norm:.4f}')


## Discussion Questions

1. Which learning rate gave the fastest decrease in training loss?
2. Which learning rate gave the best validation loss?
3. What does it mean if training loss decreases but validation loss increases?
4. Why do we call mini-batch SGD stochastic?
